In [5]:
%pip install torch torchvision



Note: you may need to restart the kernel to use updated packages.


Part 2 — Product Image Categoriser via Transfer Learning
Task 1 — Load Dataset and Create Splits

We use Fashion-MNIST (Zalando Research), the pinned canonical benchmark dataset, loaded via torchvision.datasets.FashionMNIST with automatic download. The standard split is 60,000 train / 10,000 test images; we carve a stratified 6,000-image validation split out of the training set (exceeding the brief's 5,000-image minimum), leaving 54,000 images for actual training. The test split remains completely untouched until final evaluation in Task 6.

In [6]:
import numpy as np
import torch
from torch.utils.data import Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

# Basic tensor transform for now (we'll add ImageNet normalization + resize in Task 2,
# once we've picked and confirmed the backbone's expected input size)
basic_transform = transforms.ToTensor()

# Download Fashion-MNIST (60k train, 10k test) — pinned canonical source
train_full = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=basic_transform
)
test_set = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=basic_transform
)

# Stratified validation split carved out of the 60k training set (at least 5,000 images)
labels = np.array(train_full.targets)
train_idx, val_idx = train_test_split(
    np.arange(len(train_full)),
    test_size=6000,          # gives us a clean 6,000-image validation split (>5,000 required)
    stratify=labels,
    random_state=42
)

train_set = Subset(train_full, train_idx)
val_set = Subset(train_full, val_idx)

class_names = train_full.classes  # ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                                   #  'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Train split size:", len(train_set))
print("Val split size:  ", len(val_set))
print("Test split size: ", len(test_set))
print("Classes:", class_names)

# Sanity check: confirm stratification held roughly balanced
import collections
val_labels = labels[val_idx]
print("Val class distribution:", collections.Counter(val_labels))

100%|██████████| 26.4M/26.4M [03:16<00:00, 135kB/s]   
100%|██████████| 29.5k/29.5k [00:00<00:00, 171kB/s]
100%|██████████| 4.42M/4.42M [00:48<00:00, 90.5kB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.84MB/s]
C:\Users\JH\AppData\Local\Temp\ipykernel_7276\349386210.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  labels = np.array(train_full.targets)


Train split size: 54000
Val split size:   6000
Test split size:  10000
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Val class distribution: Counter({np.int64(3): 600, np.int64(1): 600, np.int64(9): 600, np.int64(8): 600, np.int64(2): 600, np.int64(4): 600, np.int64(0): 600, np.int64(6): 600, np.int64(5): 600, np.int64(7): 600})


Task 2 — Preprocess for Pretrained Backbone

We selected ResNet-18 as the backbone (over EfficientNet-B0) for its lower compute cost on CPU, which is more than sufficient for Fashion-MNIST's relatively low visual complexity. ResNet-18 expects 3-channel, 224×224 input normalized with ImageNet statistics. Since Fashion-MNIST images are single-channel grayscale, we replicate the channel three times, resize from 28×28 to 224×224, and normalize using the standard ImageNet mean/std the backbone was originally trained with.

In [7]:
from torchvision import transforms

# ResNet-18 expected preprocessing
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

resnet_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # 1 channel -> 3 channels
    transforms.Resize((224, 224)),                  # ResNet-18's expected input size
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Re-load datasets with the real transform this time
train_full = datasets.FashionMNIST(root="./data", train=True, download=False, transform=resnet_transform)
test_set   = datasets.FashionMNIST(root="./data", train=False, download=False, transform=resnet_transform)

train_set = Subset(train_full, train_idx)
val_set   = Subset(train_full, val_idx)

# Sanity check on one image
img, label = train_set[0]
print("Image tensor shape:", img.shape)   # should be torch.Size([3, 224, 224])
print("Label:", label, "->", class_names[label])
print("Pixel value range (post-normalize):", img.min().item(), "to", img.max().item())

Image tensor shape: torch.Size([3, 224, 224])
Label: 8 -> Bag
Pixel value range (post-normalize): -2.1179039478302 to 2.5877127647399902


Task 3 — Build Transfer-Learning Model and Cache Backbone Features

We load ResNet-18 pretrained on ImageNet and freeze all of its parameters, since we are using feature extraction (not fine-tuning) as our first approach. The final classification layer is replaced with nn.Identity() so the backbone outputs raw 512-dimensional pooled features instead of 1000-class ImageNet scores.

Per the brief's speed-tip, since the backbone is frozen, we run it once over every image in train/val/test and cache the resulting feature vectors to cached_features.pt. This is mathematically identical to re-running the frozen backbone every epoch, but turns what would be an hours-long CPU training loop into a single ~30-minute feature-extraction pass, after which head-only training (Task 4) takes only seconds per epoch.

In [8]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load pretrained ResNet-18
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all backbone layers (feature extraction mode)
for param in backbone.parameters():
    param.requires_grad = False

# Remove the final classification layer — we want raw 512-dim features, not ImageNet's 1000 classes
feature_dim = backbone.fc.in_features   # 512 for ResNet-18
backbone.fc = nn.Identity()             # output raw pooled features instead of class scores
backbone = backbone.to(device)
backbone.eval()

print("Feature dimension:", feature_dim)

# --- Feature extraction / caching ---
def extract_features(dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    all_features = []
    all_labels = []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            feats = backbone(imgs)          # shape: [batch, 512]
            all_features.append(feats.cpu())
            all_labels.append(labels)
            if i % 50 == 0:
                print(f"  batch {i}/{len(loader)}")
    return torch.cat(all_features), torch.cat(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(train_set)

print("Extracting val features...")
val_features, val_labels = extract_features(val_set)

print("Extracting test features...")
test_features, test_labels = extract_features(test_set)

print("Train features shape:", train_features.shape)
print("Val features shape:  ", val_features.shape)
print("Test features shape: ", test_features.shape)

# Save cached features so we never have to redo this expensive step
torch.save({
    "train_features": train_features, "train_labels": train_labels,
    "val_features": val_features, "val_labels": val_labels,
    "test_features": test_features, "test_labels": test_labels,
}, "cached_features.pt")
print("Cached features saved to cached_features.pt")

Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\JH/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:08<00:00, 5.74MB/s]


Feature dimension: 512
Extracting train features...
  batch 0/844
  batch 50/844
  batch 100/844
  batch 150/844
  batch 200/844
  batch 250/844
  batch 300/844
  batch 350/844
  batch 400/844
  batch 450/844
  batch 500/844
  batch 550/844
  batch 600/844
  batch 650/844
  batch 700/844
  batch 750/844
  batch 800/844
Extracting val features...
  batch 0/94
  batch 50/94
Extracting test features...
  batch 0/157
  batch 50/157
  batch 100/157
  batch 150/157
Train features shape: torch.Size([54000, 512])
Val features shape:   torch.Size([6000, 512])
Test features shape:  torch.Size([10000, 512])
Cached features saved to cached_features.pt


Task 4 — Train Classifier Head (Feature Extraction Phase)

The ResNet-18 backbone stays frozen; only a small new head (512 → 128 → 10) is trained, using the cached features from Task 3 — no image ever passes through the backbone again during this phase. Documented hyperparameters: Optimizer: Adam, Learning rate: 1e-3, Batch size: 128, Epochs: 15. Per the brief, if validation accuracy falls below 80% after this phase, we proceed to fine-tuning (Task 5, unfreezing late backbone layers) — otherwise feature extraction alone is sufficient.

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Load the cached features (so this cell can be re-run independently without redoing Task 3)
cache = torch.load("cached_features.pt")
train_features, train_labels = cache["train_features"], cache["train_labels"]
val_features, val_labels     = cache["val_features"], cache["val_labels"]
test_features, test_labels   = cache["test_features"], cache["test_labels"]

# --- Documented hyperparameters (required by the brief) ---
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15
OPTIMIZER = "Adam"

print(f"Batch size: {BATCH_SIZE}, Optimizer: {OPTIMIZER}, LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")

# Wrap cached features in DataLoaders
train_ds = TensorDataset(train_features, train_labels)
val_ds   = TensorDataset(val_features, val_labels)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# --- The new classifier head ---
class ClassifierHead(nn.Module):
    def __init__(self, in_features=512, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.net(x)

head = ClassifierHead()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(head.parameters(), lr=LEARNING_RATE)

# --- Training loop ---
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for feats, labels in loader:
            outputs = model(feats)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

history = []
for epoch in range(NUM_EPOCHS):
    head.train()
    running_loss = 0.0
    for feats, labels in train_loader:
        optimizer.zero_grad()
        outputs = head(feats)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * feats.size(0)

    train_loss = running_loss / len(train_ds)
    val_acc = evaluate(head, val_loader)
    history.append((epoch + 1, train_loss, val_acc))
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train loss: {train_loss:.4f} | Val acc: {val_acc:.4f}")

final_val_acc = history[-1][2]
print(f"\nFinal feature-extraction validation accuracy: {final_val_acc:.4f}")

Batch size: 128, Optimizer: Adam, LR: 0.001, Epochs: 15
Epoch 1/15 | Train loss: 0.5326 | Val acc: 0.8740
Epoch 2/15 | Train loss: 0.3649 | Val acc: 0.8860
Epoch 3/15 | Train loss: 0.3361 | Val acc: 0.8872
Epoch 4/15 | Train loss: 0.3138 | Val acc: 0.8932
Epoch 5/15 | Train loss: 0.3002 | Val acc: 0.8918
Epoch 6/15 | Train loss: 0.2883 | Val acc: 0.8992
Epoch 7/15 | Train loss: 0.2813 | Val acc: 0.9042
Epoch 8/15 | Train loss: 0.2733 | Val acc: 0.9037
Epoch 9/15 | Train loss: 0.2644 | Val acc: 0.9032
Epoch 10/15 | Train loss: 0.2594 | Val acc: 0.9047
Epoch 11/15 | Train loss: 0.2557 | Val acc: 0.9035
Epoch 12/15 | Train loss: 0.2513 | Val acc: 0.9028
Epoch 13/15 | Train loss: 0.2439 | Val acc: 0.9043
Epoch 14/15 | Train loss: 0.2392 | Val acc: 0.9088
Epoch 15/15 | Train loss: 0.2359 | Val acc: 0.9052

Final feature-extraction validation accuracy: 0.9052


Task 4 Result — Feature Extraction Sufficient

Feature-extraction-only validation accuracy reached 90.52%, comfortably above the 80% threshold specified in the brief. Per Task 5's condition ("if validation accuracy is below 80%, unfreeze late layers"), fine-tuning was not required. We proceed directly to final test-set evaluation using the frozen-backbone + trained-head configuration.

Task 6 — Final Test-Set Evaluation

Evaluated on the untouched 10,000-image test split (never seen during training or validation, and separate from the validation split used for model selection). We report overall accuracy, the full 10×10 confusion matrix, and per-class precision/recall/F1, as required by the brief.

In [10]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

test_ds = TensorDataset(test_features, test_labels)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

head.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for feats, labels in test_loader:
        outputs = head(feats)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds)
        all_labels.append(labels)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

test_acc = accuracy_score(all_labels, all_preds)
print(f"Final TEST accuracy: {test_acc:.4f}\n")

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix (rows=true, cols=predicted):")
print(cm)

print("\nPer-class precision/recall/F1:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

Final TEST accuracy: 0.8994

Confusion matrix (rows=true, cols=predicted):
[[855   3  18  11   2   0 107   0   4   0]
 [  4 974   3  13   1   1   3   0   1   0]
 [  8   0 872   5  47   0  68   0   0   0]
 [ 35   6  14 848  22   1  73   0   1   0]
 [  1   0  56  25 826   0  90   0   2   0]
 [  0   0   0   0   0 953   1  35   1  10]
 [103   0  50  19  54   0 770   0   3   1]
 [  0   0   0   0   0  13   0 958   0  29]
 [  2   0   1   2   0   1   7   1 986   0]
 [  0   0   0   0   0   9   0  39   0 952]]

Per-class precision/recall/F1:
              precision    recall  f1-score   support

 T-shirt/top     0.8482    0.8550    0.8516      1000
     Trouser     0.9908    0.9740    0.9823      1000
    Pullover     0.8600    0.8720    0.8659      1000
       Dress     0.9187    0.8480    0.8820      1000
        Coat     0.8676    0.8260    0.8463      1000
      Sandal     0.9744    0.9530    0.9636      1000
       Shirt     0.6881    0.7700    0.7268      1000
     Sneaker     0.9274    0.

Task 7 — Confusion Pattern Analysis

T-shirt/top ↔ Shirt (210 total confusions): These are visually the most similar garment silhouettes in the entire dataset — both are upper-body garments with a horizontal hem, short/rolled sleeves in low-res 28×28 grayscale, and no strong distinguishing texture at this resolution. A Shirt typically has a collar and button placket that would distinguish it from a T-shirt in a real photo, but at 28×28 grayscale (now upsampled to 224×224, which doesn't add real detail — it just interpolates), that fine collar/button detail is largely lost, leaving mostly the overall garment outline for the model to go on.

Coat ↔ Shirt (144 total confusions): Coats and shirts share a similar boxy upper-body silhouette when photographed flat/front-on, differing mainly in sleeve length, layering bulk, and length — all cues that are subtle at low resolution. A heavier coat's slightly wider silhouette and longer hem are the main distinguishing signals, but these are easy to blur into a shirt's overall shape once downsampled, especially when garments are close in image cropping.

In [11]:
import os
import torch
from PIL import Image
import numpy as np

os.makedirs("models", exist_ok=True)
os.makedirs("data/sample_images", exist_ok=True)

# --- Save the trained head's weights ---
torch.save(head.state_dict(), "models/product_classifier.pt")
print("Saved model weights to models/product_classifier.pt")

# --- Documented loading + single-image prediction function ---
def load_classifier(weights_path="models/product_classifier.pt", device="cpu"):
    """
    Loads the ResNet-18 backbone (frozen, feature-extractor) + trained classifier head.
    Returns a function that takes an image path and returns (predicted_label, confidence).
    """
    from torchvision import models, transforms
    import torch.nn as nn

    # Rebuild backbone (same as Task 3)
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    backbone.fc = nn.Identity()
    backbone.eval().to(device)

    # Rebuild head architecture and load trained weights
    head = ClassifierHead()
    head.load_state_dict(torch.load(weights_path, map_location=device))
    head.eval().to(device)

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

    def predict(image_path):
        img = Image.open(image_path).convert("L")  # ensure grayscale in
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            feats = backbone(x)
            logits = head(feats)
            probs = torch.softmax(logits, dim=1)
            conf, pred_idx = probs.max(dim=1)
        return {
            "predicted_label": class_names[pred_idx.item()],
            "confidence": round(conf.item(), 4)
        }

    return predict

# --- Export at least 5 real test-split images as actual .png files ---
# Use the ORIGINAL (untransformed) test set so we save real viewable images, not normalized tensors
raw_test_set = datasets.FashionMNIST(root="./data", train=False, download=False, transform=None)

sample_indices = [0, 1, 2, 3, 4, 5, 6]  # first 7 test images, covering a spread of classes
for i, idx in enumerate(sample_indices):
    img, label = raw_test_set[idx]  # PIL Image, int label
    label_name = class_names[label].replace("/", "-")  # avoid slash in filename
    filename = f"data/sample_images/{i:02d}_{label_name}.png"
    img.save(filename)
    print(f"Saved {filename}")

# --- Verify the saved model + loader work end-to-end ---
predict_fn = load_classifier()
test_image_path = f"data/sample_images/00_{class_names[raw_test_set[0][1]].replace('/', '-')}.png"
result = predict_fn(test_image_path)
print(f"\nSanity check on {test_image_path}:")
print(f"True label: {class_names[raw_test_set[0][1]]}")
print(f"Predicted: {result}")

Saved model weights to models/product_classifier.pt
Saved data/sample_images/00_Ankle boot.png
Saved data/sample_images/01_Pullover.png
Saved data/sample_images/02_Trouser.png
Saved data/sample_images/03_Trouser.png
Saved data/sample_images/04_Shirt.png
Saved data/sample_images/05_Trouser.png
Saved data/sample_images/06_Coat.png

Sanity check on data/sample_images/00_Ankle boot.png:
True label: Ankle boot
Predicted: {'predicted_label': 'Ankle boot', 'confidence': 0.9994}


Task 8 — Save Artifacts for Part 3

Saves the trained classifier head to models/product_classifier.pt (the backbone itself doesn't need saving since it's the standard pretrained ResNet-18, reloaded fresh in the loader function). Exports 7 real test-split images as .png files to data/sample_images/, named with their true label for traceability. Includes a documented load_classifier() function that Part 3's classify_product_image tool will call directly — this is verified end-to-end with a sanity check against a known label.

In [12]:
import os

# Rename existing files to remove spaces
for fname in os.listdir("data/sample_images"):
    if " " in fname:
        new_fname = fname.replace(" ", "-")
        os.rename(f"data/sample_images/{fname}", f"data/sample_images/{new_fname}")
        print(f"Renamed: {fname} -> {new_fname}")

Renamed: 00_Ankle boot.png -> 00_Ankle-boot.png
